# ProScore 真实数据全流程测试（GMSC / Home Credit / Lending Club）

本 Notebook 对应文档：[real-data-test-plan.md](../docs/使用指南/real-data-test-plan.md)

**说明**：在部分网络环境 Zenodo/Kaggle 可能无法直连；请在本机终端执行 `scripts/download_real_data.py` 下载数据后，再运行本 Notebook。

**三条路径**：分模块 API → 链式 API → Excel（需 `pip install "proscore[excel]"`）。


## 0. 环境与项目根目录


In [1]:
import subprocess
import sys
from pathlib import Path

# 项目根（支持在 notebooks/ 或仓库根目录打开）
_cwd = Path.cwd()
for base in (_cwd, _cwd.parent):
    if (base / "src" / "proscore").is_dir():
        PROJECT_ROOT = base.resolve()
        break
else:
    raise FileNotFoundError("请在 ProScore 仓库根目录或 notebooks/ 下打开本 Notebook")

_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

DATA_RAW = PROJECT_ROOT / "data" / "gmsc_train.csv"
DATA_CSV = PROJECT_ROOT / "data" / "processed" / "real_scorecard.csv"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "bad_flag"
DATE_COL = "apply_date"

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /Users/pro/Desktop/proscore


## 1. 下载与预处理（本机执行）

若 `data/gmsc_train.csv` 不存在，在终端运行：

```bash
cd <仓库根>
python scripts/download_real_data.py gmsc
python scripts/prepare_real_scorecard_data.py --sample 50000
```

下面单元在 Notebook 内尝试自动执行（失败则请手动运行上述命令）。


In [2]:
import subprocess

if not DATA_RAW.exists():
    r = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "download_real_data.py"), "gmsc"],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    print("download gmsc:", r.returncode, r.stderr[-500:] if r.stderr else "")

if not DATA_CSV.exists() or DATA_CSV.stat().st_size < 1000:
    r = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_real_scorecard_data.py"), "--sample", "50000"],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    print("prepare:", r.returncode, r.stdout[-800:] if r.stdout else "", r.stderr[-500:] if r.stderr else "")

assert DATA_CSV.exists(), f"缺少 {DATA_CSV}，请先运行 scripts/prepare_real_scorecard_data.py"
print("数据就绪:", DATA_CSV)


数据就绪: /Users/pro/Desktop/proscore/data/processed/real_scorecard.csv


## 2. 加载数据与随机切分

按 **目标变量分层** 随机划分 Train / Test / OOT（**不依赖** `apply_date` 年份），适合 Lending Club 单年切片、GMSC 伪日期等。比例见代码中 `REST_FRAC`、`OOT_OF_REST`。


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_CSV, parse_dates=[DATE_COL])
assert TARGET in df.columns

# 随机分层：先划出「非训练」池，再从中对半拆成 Test 与 OOT（与 apply_date 年份无关）
REST_FRAC = 0.30  # 非训练池占比 → Train 约 70%
OOT_OF_REST = 0.50  # 非训练池中一半为 Test、一半为 OOT

train_df, rest = train_test_split(
    df,
    test_size=REST_FRAC,
    stratify=df[TARGET],
    random_state=42,
)
test_df, oot_df = train_test_split(
    rest,
    test_size=OOT_OF_REST,
    stratify=rest[TARGET],
    random_state=42,
)

train_m = train_df.drop(columns=[DATE_COL])
test_m = test_df.drop(columns=[DATE_COL])
oot_m = oot_df.drop(columns=[DATE_COL])

print("切分: 随机分层（stratify=bad_flag）")
print("Train", train_m.shape, "bad_rate", train_m[TARGET].mean())
print("Test ", test_m.shape, "bad_rate", test_m[TARGET].mean())
print("OOT  ", oot_m.shape, "bad_rate", oot_m[TARGET].mean())


切分: 随机分层（stratify=bad_flag）
Train (1855, 67) bad_rate 0.13153638814016172
Test  (397, 67) bad_rate 0.1309823677581864
OOT   (398, 67) bad_rate 0.1306532663316583


## 3. 分模块 API（与链式顺序一致）

### 为何三条路径结果可能不一致？

| 原因 | 分模块 vs 链式 | Excel（模板默认） |
|------|----------------|-------------------|
| 数据切分 | 同一 `train_m` / `test_m` / `oot_m` | 仅 **Train/Test** 两集（默认 70/30）；**无第三份 OOT**，`evaluate` 里 OOT 相关指标可能缺失或与 Notebook 不可比 |
| 超参数 | 见下方代码显式传入 | 曾默认：`prefilter` 未传 `iv_range` 时用 Filter 默认 IV 筛、`max_corr=0.7`；分箱 `chi`；逐步 `max_iter_round=100`；`n_min/n_max` 5～12 |
| 随机性 | `random_state=42` 一致 | `Global.random_seed` 控制切分与逐步（默认 42） |

**对齐做法**：链式/分模块与 Excel 使用相同超参；Excel 的 **Screening / Binning / Modeling** 表与 Notebook 对齐；接受 Excel 为 **两集切分** 或扩展脚本支持三集（未做）。下方 Excel 单元已写入与链式一致的常用参数。


In [4]:
import time
from proscore import inspect
from proscore.binning import Binning
from proscore.evaluate import evaluate
from proscore.modeling import ScoreCard
from proscore.selection import Filter, StepwiseSelector
from proscore.transform import WOETransformer

t0 = time.perf_counter()
_ = inspect.detect(train_m, target=TARGET)

feat0 = [c for c in train_m.columns if c != TARGET and pd.api.types.is_numeric_dtype(train_m[c])]
pf = Filter(max_corr=0.9, max_vif=15, iv_range=None, max_psi=None)
pf.fit(
    train_m[feat0],
    train_m[TARGET],
    X_test=test_m[feat0] if len(test_m) else None,
    bin_table=None,
)
feat1 = pf.support_

X_fit = pd.concat([train_m[feat1], train_m[[TARGET]]], axis=1)
binner = Binning(method="frequency", n_bins=5)
binner.fit(X_fit, y=TARGET)

rf = Filter(iv_range=(0.02, None), max_corr=0.9, max_vif=15)
rf.fit(
    train_m[feat1],
    train_m[TARGET],
    X_test=test_m[feat1],
    bin_table=binner.bin_table_,
)
feat2 = rf.support_

wt = WOETransformer(unseen_strategy="worst")
tables = {k: v for k, v in binner.bin_table_.items() if k in feat2}
wt.fit(tables)

train_woe = wt.transform(train_m[feat2])
train_woe[TARGET] = train_m[TARGET].values
test_woe = wt.transform(test_m[feat2]) if len(test_m) else None

ss = StepwiseSelector(n_min=3, n_max=8, force_fill=True, max_iter_round=6)
ss.fit(
    train_woe,
    train_m[TARGET],
    candidates=feat2,
    X_test=test_woe,
    y_test=test_m[TARGET].values if test_woe is not None else None,
)

sc = ScoreCard(odds=20, pdo=20, base_score=600)
sc.fit(train_woe, y=TARGET, features=ss.support_)
sel_tables = {k: v for k, v in binner.bin_table_.items() if k in ss.support_}
sc.scorecard(sel_tables)

oot_woe = wt.transform(oot_m[ss.support_]) if len(oot_m) else None
mod_eval = evaluate(
    sc.model_,
    train_woe[ss.support_],
    train_m[TARGET],
    X_test=test_woe[ss.support_] if test_woe is not None else None,
    y_test=test_m[TARGET] if test_woe is not None else None,
    X_oot=oot_woe,
    y_oot=oot_m[TARGET] if oot_woe is not None else None,
    features=ss.support_,
)

mod_support = list(ss.support_)
mod_time = time.perf_counter() - t0
print("分模块耗时(s):", round(mod_time, 2))
print("入模变量:", mod_support)
print("evaluate keys:", sorted(k for k in mod_eval if "ks" in k or "auc" in k))
for k in sorted(mod_eval):
    if "ks" in k or "auc" in k:
        print(f"  {k}: {mod_eval[k]}")


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


分模块耗时(s): 117.92
入模变量: ['grade_ord', 'mo_sin_old_il_acct', 'installment', 'open_acc_6m', 'inq_last_12m', 'all_util', 'num_tl_90g_dpd_24m', 'mths_since_recent_bc']
evaluate keys: ['ks_reduce', 'ks_rel_gap', 'oot_auc', 'oot_ks', 'test_auc', 'test_ks', 'trn_auc', 'trn_ks']
  ks_reduce: 0.157239
  ks_rel_gap: 0.40207
  oot_auc: 0.695281
  oot_ks: 0.292241
  test_auc: 0.615831
  test_ks: 0.233835
  trn_auc: 0.735006
  trn_ks: 0.391074


## 4. 链式 API


In [5]:
import proscore as ps

t0 = time.perf_counter()
p = (
    ps.ProScore()
    .read(train=train_m, test=test_m, oot=oot_m, target=TARGET)
    .detect()
    .prefilter(max_corr=0.9, max_vif=15, iv_range=None, max_psi=None)
    .bin(method="frequency", n_bins=5)
    .refine(iv_range=(0.02, None), max_corr=0.9, max_vif=15)
    .transform(unseen_strategy="worst")
    .select(n_min=3, n_max=8, force_fill=True, max_iter_round=6)
    .fit(odds=20, pdo=20, base_score=600)
    .scorecard()
    .evaluate()
)
chain_time = time.perf_counter() - t0
print("链式耗时(s):", round(chain_time, 2))
print("入模变量:", p.support_)
print("与分模块入模是否相同:", set(p.support_) == set(mod_support))
er = {k: p.eval_result[k] for k in p.eval_result if "ks" in k or "auc" in k}
print("evaluate:", er)


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


链式耗时(s): 65.9
入模变量: ['grade_ord', 'mo_sin_old_il_acct', 'installment', 'open_acc_6m', 'inq_last_12m', 'all_util', 'num_tl_90g_dpd_24m', 'mths_since_recent_bc']
与分模块入模是否相同: True
evaluate: {'trn_ks': 0.391074, 'trn_auc': 0.735006, 'test_ks': 0.233835, 'test_auc': 0.615831, 'ks_reduce': 0.157239, 'ks_rel_gap': 0.40207, 'oot_ks': 0.292241, 'oot_auc': 0.695281}


## 5. Excel 配置驱动

需安装：`pip install "proscore[excel]"`。

填写 **Data** 表后调用 `python -m proscore run ...`（工作目录为项目根，以便生成报告目录）。

与上一节一致：**`dev_start` / `dev_end` / `oot_start` 留空**，由 Excel 内在全量样本上按 `train_ratio` **随机**划分 Train/Test；不设时间型 OOT（与 Notebook 的「三份随机 OOT」不完全相同，但同为非时间切分）。


In [ ]:
import shutil
import tempfile

try:
    import openpyxl  # noqa: F401
except ImportError:
    print('未安装 openpyxl，跳过 Excel 路径。请执行: pip install "proscore[excel]"')
else:
    tmp = Path(tempfile.mkdtemp(dir=OUTPUT_DIR))
    subprocess.run(
        [sys.executable, "-m", "proscore", "template", str(tmp)],
        cwd=str(PROJECT_ROOT),
        check=True,
    )
    xlsx = tmp / "pipeline_template.xlsx"
    sheets = {n: pd.read_excel(xlsx, sheet_name=n, engine="openpyxl") for n in pd.ExcelFile(xlsx).sheet_names}

    for name, sdf in sheets.items():
        if "您的取值" in sdf.columns:
            sdf["您的取值"] = sdf["您的取值"].astype(object)

    d = sheets["Data"]
    d.loc[d["参数名"] == "data_file", "您的取值"] = str(DATA_CSV.resolve())
    d.loc[d["参数名"] == "target", "您的取值"] = TARGET
    d.loc[d["参数名"] == "time_col", "您的取值"] = DATE_COL
    # 留空：不按日期截开发池 / 不切时间 OOT；PipelineConfig 在全量上丢掉时间列后随机分 Train/Test
    for _key in ("dev_start", "dev_end", "oot_start", "oot_end"):
        if (d["参数名"] == _key).any():
            d.loc[d["参数名"] == _key, "您的取值"] = ""

    # 模板 Variables 中的示例变量（income 等）在真实数据中不存在，需清空以免校验失败
    vsh = sheets["Variables"]
    sheets["Variables"] = vsh.head(0).copy()

    g = sheets["Global"]
    if "参数名" in g.columns and "您的取值" in g.columns:
        g["您的取值"] = g["您的取值"].astype(object)
        if (g["参数名"] == "project_name").any():
            g.loc[g["参数名"] == "project_name", "您的取值"] = "real_nb_excel"

    filled = tmp / "real_pipeline.xlsx"
    with pd.ExcelWriter(filled, engine="openpyxl") as w:
        for name, sdf in sheets.items():
            sdf.to_excel(w, sheet_name=name, index=False)

    r = subprocess.run(
        [
            sys.executable,
            "-m",
            "proscore",
            "run",
            str(filled),
            "--output-script",
            str(tmp / "generated_from_excel.py"),
        ],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    print("Excel run returncode:", r.returncode)
    if r.stdout:
        print(r.stdout[-1200:])
    if r.stderr:
        print("stderr:", r.stderr[-800:])
    rep = PROJECT_ROOT / "real_nb_excel_report" / "report.md"
    print("报告存在:", rep.exists(), rep)


模板已生成: /Users/pro/Desktop/proscore/notebooks/output/tmpkp8x1cee/pipeline_template.xlsx
Excel run returncode: 0
脚本已生成: /Users/pro/Desktop/proscore/notebooks/output/tmpkp8x1cee/generated_from_excel.py

  real_nb_excel — 建模完成
  入模变量: tot_hi_cred_lim | revol_util | mths_since_rcnt_il | delinq_2yrs | total_bc_limit | grade_ord | funded_amnt_inv | dti
  Train KS: 0.4815
  Test  KS: 0.2143
  Train AUC: 0.7884
  Test  AUC: 0.6130
  PSI(train vs test): 0.0142


报告存在: True /Users/pro/Desktop/proscore/real_nb_excel_report/report.md


## 7. 模型诊断（新增功能 v0.2）

在任意完整 pipeline 结束后调用 `.diagnose()` 可获得结构化健康检查（区分力 / 过拟合 / 稳定性 / 变量质量 4 层，含根因变量定位）。

支持：
- 链式：`p.diagnose(print_report=False)` → 通过 `p.diagnosis_` 访问
- ReportBuilder 自动附带（`from_proscore` 已增强）
- 独立使用 `from proscore.evaluate import diagnose`


In [ ]:
# 以链式 API 产出的 p 为例（Excel 模式同理，把 p3 换成对应变量）
if "p" in globals() and getattr(p, "eval_result", None):
    p.diagnose(print_report=False)          # 静默生成
    print("【诊断结果】严重:", len(p.diagnosis_.critical),
          "警告:", len(p.diagnosis_.warnings),
          "提示:", len(p.diagnosis_.infos))

    # 演示 ReportBuilder 自动包含诊断章节（C3 增强）
    from proscore.report import ReportBuilder
    rb = ReportBuilder.from_proscore(p)
    md = rb.build()
    print("ReportBuilder 自动带「模型诊断」章节:", "## 模型诊断" in md)

    # 如需保存完整报告（含诊断）
    # rb.save("real_data_diagnosis_report.md")
else:
    print("请先运行上面的链式/分模块/Excel 任意一个完整流程，再执行本单元")


## 6. 小结

- 对比 **分模块** 与 **链式** 的 `support_` 与 `trn_ks` / `test_ks` / `oot_ks`。
- Excel 路径以 **report.md** 是否生成为准；模板默认分箱为 `chi`，与上方代码的 `frequency` 可能略有差异，属预期。

更换为 **Home Credit** 或 **Lending Club**：将原始 CSV 放到 `data/` 对应子目录后运行 `python scripts/prepare_real_scorecard_data.py`（自动探测源），再重新运行本 Notebook。
